# Processamento de Língua Natural - EP (Testes)
Baseline: Regressão Logística com contagem TF-IDF

## Bibliotecas
Sklearn, Pandas

In [28]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterGrid, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

## Importação do Conjunto de Treino
O Pandas importa ambos os conjuntos de dados (treino e teste) como strings, para evitar erros de tipagem no momento da tokenização.

In [29]:
dados_treino = pd.read_excel("dados/train.xlsx", converters={'resp_text':str,'clarity':str})

X_treino, Y_treino = dados_treino["resp_text"].tolist(), dados_treino["clarity"].tolist()

In [30]:
dados_treino

,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234
...,...,...
20087,Prezado Sr. Luciano Wolff A sua solicitação ...,c1
20088,"PREZADO CIDADãO, A COORDENAçãO DO PROGRAMA FA...",c1
20089,"Prezado, Informamos que o Ministério da Agr...",c1
20090,"Prezado Senhor, Encaminhamos em anexo respos...",c1


## Tokenização (TF-IDF)

In [31]:
# Tokenizador TF-IDF
vect = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode'
)

## Classificador (Regressão Logística)

In [32]:
# Classificador base usado na busca em grid
clf = LogisticRegression(
    max_iter=100,
    solver='saga',
    tol=1e-3,
    random_state=42
)

## Busca em Grid e Validação Cruzada

In [33]:
# Hiperparâmetros avaliados
param_grid = {
    'C': [0.05, 0.1, 0.2, 0.5, 1.0],
    'class_weight': [None, 'balanced']
}

# Usa 5 folds para selecionar os hiperparâmetros.
# O melhor conjunto é reavaliado posteriormente com 10 folds.
cv_grid = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

X_array = np.array(X_treino, dtype=object)
Y_array = np.array(Y_treino, dtype=object)
configuracoes = list(ParameterGrid(param_grid))

train_scores = np.zeros((len(configuracoes), cv_grid.get_n_splits()))
validation_scores = np.zeros((len(configuracoes), cv_grid.get_n_splits()))

# O TF-IDF é ajustado somente nos dados de treino de cada fold.
# Todas as configurações da regressão logística usam a mesma representação do fold.
for fold, (idx_treino, idx_validacao) in enumerate(cv_grid.split(X_array, Y_array)):
    vect_fold = clone(vect)

    X_fold_treino = vect_fold.fit_transform(X_array[idx_treino])
    X_fold_validacao = vect_fold.transform(X_array[idx_validacao])

    for i, params in enumerate(configuracoes):
        clf_fold = clone(clf).set_params(**params)
        clf_fold.fit(X_fold_treino, Y_array[idx_treino])

        train_scores[i, fold] = clf_fold.score(X_fold_treino, Y_array[idx_treino])
        validation_scores[i, fold] = clf_fold.score(X_fold_validacao, Y_array[idx_validacao])

# Relatório das 10 configurações testadas
relatorio_grid = pd.DataFrame([
    {
        **params,
        'acuracia_treino_media': train_scores[i].mean(),
        'acuracia_validacao_media': validation_scores[i].mean(),
        'desvio_padrao_validacao': validation_scores[i].std(),
        'gap_treino_validacao': train_scores[i].mean() - validation_scores[i].mean()
    }
    for i, params in enumerate(configuracoes)
])

relatorio_grid = relatorio_grid.sort_values(
    by='acuracia_validacao_media',
    ascending=False
).reset_index(drop=True)

relatorio_grid.index = relatorio_grid.index + 1
relatorio_grid.index.name = 'rank'

top_10 = relatorio_grid.head(10)
display(top_10)

# Salva o relatório para documentação do experimento
top_10.to_csv("grid_top10.csv")

melhores_parametros = {
    'C': top_10.iloc[0]['C'],
    'class_weight': top_10.iloc[0]['class_weight']
}

print("Melhores hiperparâmetros:", melhores_parametros)

# Reavalia o vencedor com 10 folds, mantendo comparabilidade com o baseline anterior
clf_final = clone(clf).set_params(**melhores_parametros)

pipeline = Pipeline([
    ('tfidf', clone(vect)),
    ('clf', clf_final)
])

cv_final = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

resultados = cross_validate(
    pipeline,
    X_treino,
    Y_treino,
    scoring='accuracy',
    cv=cv_final,
    return_train_score=True
)

acuracia_validacao = resultados['test_score'].mean()
acuracia_treino_cv = resultados['train_score'].mean()

print(f"Acurácia Média da Validação Cruzada (10 folds): {acuracia_validacao:.2%}")
print(f"Acurácia Média de Treino nos Folds: {acuracia_treino_cv:.2%}")
print(f"Gap Treino-Validação: {(acuracia_treino_cv - acuracia_validacao):.2%}")

,C,class_weight,acuracia_treino_media,acuracia_validacao_media,desvio_padrao_validacao,gap_treino_validacao
rank,,,,,,
1,0.50,balanced,0.642091,0.462473,0.008287,0.179618
2,0.50,None,0.640216,0.461179,0.008089,0.179037
3,0.20,balanced,0.580137,0.459984,0.007434,0.120153
4,1.00,balanced,0.693477,0.459089,0.009329,0.234388
5,1.00,None,0.692564,0.457695,0.009089,0.234869
6,0.20,None,0.577112,0.456699,0.007089,0.120413
7,0.10,balanced,0.540840,0.455903,0.007828,0.084937
8,0.10,None,0.538169,0.453862,0.007218,0.084307
9,0.05,balanced,0.507947,0.447940,0.007408,0.060007


Melhores hiperparâmetros: {'C': np.float64(0.5), 'class_weight': 'balanced'}
Acurácia Média da Validação Cruzada (10 folds): 46.25%
Acurácia Média de Treino nos Folds: 64.21%
Gap Treino-Validação: 17.96%


## Treino do Modelo

In [34]:
pipeline.fit(X_treino, Y_treino)
print(f"Acurácia do Treino (modelo final): {pipeline.score(X_treino, Y_treino):.2%}")

Acurácia do Treino (modelo final): 63.70%


## Execução do Teste e Salvamento do Resultado

In [35]:
# Importação dos dados de teste
dados_teste = pd.read_excel("dados/test1.xlsx", converters={'resp_text':str,'clarity':str})
X_teste = dados_teste["resp_text"].tolist()

# Aplicação do modelo no teste
Y_previsto = pipeline.predict(X_teste)

In [36]:
teste_rotulado = pd.DataFrame({
    'resp_text': X_teste,
    'clarity': Y_previsto
})

teste_rotulado.to_excel("teste_rotulado.xlsx", index=False)